# The Generative AI


## Structure
- What is Generative AI?
- How can you use it?
- Going Beyond ChatGPT: API & Functions
- Langchain & Beyond: Using LLMs in Applications
- Shortcomings
- Further Reading

This notebook is designed to show some of the many different ways of leveraging the power of advanced models like OpenAI's ChatGPT through their APIs rather than simply a web interface.

#### Requirements 

As this field is evolving at an extremely rapid pace (e.g. OpenAI has only recently deprecated several of their model endpoints), ensuring stability with such tools can be tricky. The following package versions below, __when run in a Colab environment__, yield consistent results.

In [2]:
!pip install openai==1.54.4 \
langchain==0.3.8 \
langchain-openai==0.2.9 \
langchain-community==0.3.8 \
pypdf==5.1.0 \
chromadb==0.5.20 \
tiktoken==0.8.0 \
huggingface-hub[hf_transfer]==0.26.2 \
ctransformers[cuda]==0.2.27 \
diffusers==0.31.0 \
llama-cpp-python==0.3.2 \
"protobuf<5.0.0"

zsh:1: no matches found: huggingface-hub[hf_transfer]==0.26.2


#### Llama weights

Towards the end of this notebook, local inference is run on a highly quantized version of Meta's LLama 2 model. The weights required to perform this are large (~4GB) so downloading them from HuggingFace before you attempt the rest of the notebook will save waiting later on. 

In [1]:
!HF_HUB_ENABLE_HF_TRANSFER=1 \
huggingface-cli download TheBloke/Llama-2-7b-Chat-GGUF \
llama-2-7b-chat.Q4_K_M.gguf --local-dir .

Traceback (most recent call last):
  File "/Users/arnaud/miniconda3/envs/nlptest/lib/python3.12/site-packages/huggingface_hub/file_download.py", line 358, in http_get
    import hf_transfer  # type: ignore[no-redef]
    ^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'hf_transfer'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/arnaud/miniconda3/envs/nlptest/bin/huggingface-cli", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/Users/arnaud/miniconda3/envs/nlptest/lib/python3.12/site-packages/huggingface_hub/commands/huggingface_cli.py", line 57, in main
    service.run()
  File "/Users/arnaud/miniconda3/envs/nlptest/lib/python3.12/site-packages/huggingface_hub/commands/download.py", line 153, in run
    print(self._download())  # Print path to downloaded files
          ^^^^^^^^^^^^^^^^
  File "/Users/arnaud/miniconda3/envs/nlptest/lib/python3.12/site-packages/huggingface_hub/commands/

#### OpenAI API key

Additionally, you will need an OpenAI API key to run many of the below examples. You can sign up for one [here](https://openai.com/blog/openai-api). The examples presented here will cost only a few cents to run!

In [ ]:
openai_api_key = 'your-openai-api-key'

## What is Generative AI?

<img src="pics/Gen_AI_players.png" width="500"/>

Gen AI is an umbrella term:
- Model is trained
- (Optional) Model fine-tuned
- Inference is run
- Images, text, sound

<img src="pics/uni_multi_gen.png" width="800"/>

## How can you use it?

The simplest (and most widely known) way to interact with high-quality generative AI is through ChatGPT:

- Trained on a vast amount of data
- 175+ billion trained parameters
- 700,000 dollars inference/ day (on top of 2-5 million dollars estimated cost for each training)

<img src="pics/methods.svg" width="1000"/>

### Pre-trained vs fine-tuning vs from-scratch

<img src="pics/fine-tuning-steps-cleaned.png" width="1000"/>

### Prompt engineering:

Some key points:

- Using role-playing
- Being specific in the task
- Highlighting inputs and specifying outputs
- "Zero-shot" vs "few-shot"
- Using Chain-of-thought prompting

### 1. Using the OpenAI API

In [ ]:
# Initialize the OpenAI API client and set your API key

import openai

openai.api_key = openai_api_key

In [ ]:
# Prompt for the AI model
prompt = "Translate the following English text to French: 'Hello, how are you?'"

# Make a request to the API to generate text
response = openai.chat.completions.create(
    model="gpt-3.5-turbo",  # Use the engine of your choice
    messages = [{"role": "user", "content": prompt}],
    max_tokens = 50
)

In [ ]:
response.choices[0].message.content

#### System prompts 

In [ ]:
# Prompt for the AI model
system_prompt = "You are a sassy culinary instructor that gives sarcastic replies"
prompt = "Give instructions to cook vegetable samosas"

# Make a request to the API to generate text
response = openai.chat.completions.create(
    model="gpt-3.5-turbo",  # Use the engine of your choice
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}],
    max_tokens = 50
)

In [ ]:
response.choices[0].message.content

#### Function calling

Imagine you have a python function:

```python
def get_current_weather(location, unit):
    ### A request is made to an API with a specific format
    ### returns some result
```

You want your user to write a question in natural language, and use that input to call the function to get the current weather.

In [ ]:
# Example user input
user_question = "I'm interested in the weather in Bozeman. I'm old-school so I like it in F?"

In [ ]:
# Use GPT to interpret the user's question
# and return the function arguments
completion = openai.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": user_question}],
    functions=[
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city with its accompanying state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string",
                         "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
],
function_call="auto",
)

In [ ]:
completion.choices[0].message.function_call.arguments

#### A worked example leveraging OpenAI and a local DataFrame

In [ ]:
import pandas as pd
import json

df = pd.read_csv("data/results.csv")

df["date"] = pd.to_datetime(df["date"])
df.head()

Write a function to retrieve specified data: all matches in a country from the starting year to the end year:

In [ ]:
def matches_finder(country: str, start_year: int, end_year: int):
    return df.loc[
        (df["country"] == country) &
        (start_year <= df["date"].dt.year) &
        (df["date"].dt.year <= end_year)
    ]

In the cell below, we describe a function that might be used to query our DataFrame. Feel free to change the `"user"` prompt in the `messages` list.

In [ ]:
query = "Tell me about matches that took place in Italy between 1980 up until the end of the 20th century"

completion = openai.chat.completions.create(
    model="gpt-4-0613",
    messages=[{"role": "user", "content": query}],
    functions=[
    {
        "name": "get_matches",
        "description": "Return the rows in a DataFrame about women's football games which satisfy the criteria",
        "parameters": {
            "type": "object",
            "properties": {
                "country": {
                    "type": "string",
                    "description": "The name of the country the matches took place e.g. France or China",
                },
                "start_year": {
                    "type": "number",
                    "description": "The year to begin filtering from e.g. 1956",
                },
                "end_year": {
                    "type": "number",
                    "description": "The year to end filtering on e.g. 2005"}
            },
            "required": ["location", "start_year", "end_year"],
        },
    }
],
function_call="auto",
)

Converting the response to something we can pass into a locally defined function. 

In [ ]:
args = json.loads(completion.choices[0].message.function_call.arguments)
args

#### Using arguments from our OpenAI Function call to interact with our locally defined function/ DataFrame

In [ ]:
matches_finder(**args)

## Langchain and Beyond:

<img src="pics/lchain.png" width="600"/>

### How can I work with larger amounts of data?

- We saw in the Transformers lecture how tricky it is to have large context windows (a.k.a. sequence length)
- ChatGPT and other models have ~32k tokens max
- Does that mean that we can only ever work with documents <32k tokens ?

We can use a Vector DataBase to store our embeddings !

### 2. Working with embeddings and larger documents

In [ ]:
# Creating embeddings
model = "text-embedding-ada-002"

embedding = openai.embeddings.create(input=["""This is a simple embedding of a sentence"""],
                                     model=model)

# How large are the embeddings we got?

import numpy as np

np.array(embedding["data"][0]["embedding"]).shape

Here, we download a book in PDF form that we can then use Langchain's document loader to prepare it for embedding

In [ ]:
! wget -O book.pdf "https://greenteapress.com/thinkpython2/thinkpython2.pdf"

In [ ]:
from langchain.document_loaders.pdf import PyPDFLoader

loader = PyPDFLoader("book.pdf")

data = loader.load()

To work with a large document, we need to split it into smaller chunks with one of Langchain's `text_splitter`s

In [ ]:
import numpy as np

print (f'You have {len(data)} documents in your data')
print (f'''There are ~{np.mean([len(x.page_content) for x in data])} characters per document''')

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)

texts = text_splitter.split_documents(data)

Next, we embed our documents directly into an in-memory vector database:

In [ ]:
from langchain.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings

vector_db = Chroma.from_documents(texts,
                                  OpenAIEmbeddings(openai_api_key = openai_api_key,
                                                   model="text-embedding-ada-002"))

<img src="pics/biencoder-diagram.png" width="600"/>


We can then embed a sentence (e.g. a question) and see which of our texts are most similar to it.

In [ ]:
# Querying the data
query = "How do I establish a Class?"
num_closest_docs = 5
docs = vector_db.similarity_search(query, k = num_closest_docs)
for k in range(num_closest_docs):
    print(f"""\n ~~~~~ Showing document #{k+1} ~~~~~ \n""")
    print(docs[k].page_content)

If we want, we can go further, passing this retrieved text as context for a prompt which we can then do question-answering on. Using `verbose = True` will allow you to see the chain of events taking place under the hood.

With Langchain, these pre-defined prompts can be altered for whatever purpose necessary.

In [ ]:
from langchain.llms import OpenAI
from langchain.chains.question_answering import load_qa_chain

llm = OpenAI(temperature=0,
             openai_api_key=openai_api_key,
             model = "gpt-3.5-turbo-instruct")

chain = load_qa_chain(llm,
                      chain_type="map_reduce",
                     verbose = True)

🔎 A note on [temperature](https://blog.lukesalamone.com/posts/what-is-temperature/) and on ["map_reduce"](https://github.com/hwchase17/langchain-hub/blob/master/chains/question_answering/map-reduce/chain.json)!

In [ ]:
query = "How do I define a class in Python"

docs = vector_db.similarity_search(query,
                                   k=5)
docs

In [ ]:
chain.run(input_documents=docs, question=query)

### 3. Running large LLMs locally w/ quantization

#### Why might you need to do this?

- Data privacy
- Fine-tuning on specific datasets
- We can even download quantized (reduced) versions of very large models from HuggingFace: https://huggingface.co/TheBloke/Llama-2-13B-GGML 😮

#### Why Quantize?

Assuming weights are stored in 32-bit float format:

1 model parameter = 4 bytes

1 billion parameters = 4 x 1,000,000,000 bytes = 4 GB (not even counting optimizer, gradient and activation info)

Many cutting edge models (Falcon, Llama, GPT 4) easily break 70 billion trainable parameters 🤯

To run these cells, __Colab with a GPU enabled is strongly recommended__, as it will significantly speed up inference times. What we are doing here is taking a version of Meta's Llama 2 model that has been significantly reduced in size and running inference on it entirely locally (simply by loading its weights onto a GPU)!

In [ ]:
from llama_cpp import Llama
llm = Llama(model_path="llama-2-7b-chat.Q4_K_M.gguf", verbose=False)

In [ ]:
output = llm("Q: How large is the earth's diameter? A: ",
             max_tokens=40,
             echo=True)
output["choices"][0]["text"]

### 4. Diffusion

Finally, we demonstrate usage of Stable Diffusion - an open source alternative to the likes of Dall-E 2 and MidJourney. Again, Colab w/ GPU is strongly recommended for faster inference here. 

See the comments about using `torch.float32` and `.to("cuda")` for implementations without GPU.

In [ ]:
from diffusers import AutoPipelineForText2Image
import torch

pipeline = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16, # Change to float32 if running without GPU, float 16 for GPU
    use_safetensors=True
).to("cuda") # If not using a GPU, remove the .to("cuda")

In [ ]:
prompt = "A Renaissance painting of the Eiffel tower" # The prompt can be changed here
pipeline(prompt, num_inference_steps=30).images[0] # Change the number of inference steps for variations

## Shortcomings

- Bias in the model
- Reliance on LLMs for labelling
- Reliability (even with the Functions API)
- Recency of data
- Confidence intervals (or lack thereof)


## Further Readings

- [OpenAI API Docs](https://platform.openai.com/docs/concepts): Filled with code examples to use
- [Andrew Ng's Prompt Engineering](https://www.deeplearning.ai/short-courses/chatgpt-prompt-engineering-for-developers/) for Developers: Excellent, free 1-hour course
- [HuggingFace Blog Post on QLora](https://huggingface.co/blog/4bit-transformers-bitsandbytes)